# SEAWAT Tutorial 2: Sea-Level Rise and the Self-Reversal of a Saltwater Wedge
### SEAWAT FloPy Short Course

This notebook reuses the Henry notebook to rebuild the confined aquifer case of Chang et al. (2011), *Does sea-level rise have an impact on saltwater intrusion?* (Adv. Water Resour. 34, 1283–1291). The sea level is raised instantly from 30 m to 34 m. The wedge first moves inland and then returns to its original position. The step numbers follow the Henry notebook. Only the changes from the Henry notebook are explained here.

## Step 0. Setting up the environment
Same as the Henry notebook. Run these cells once every time you connect.

In [ ]:
# This cell fixes a Colab problem in which the Run Model cell never finishes,
# caused by a version conflict between Colab's Python 3.13 and the Jupyter client.

import warnings, datetime, jupyter_client.session
warnings.filterwarnings("ignore", category=DeprecationWarning)                        # hide the warnings
jupyter_client.session.utcnow = lambda: datetime.datetime.now(datetime.timezone.utc)  # replace the warning-producing function with a warning-free version

import ipykernel.kernelbase
ipykernel.kernelbase.now = jupyter_client.session.utcnow                              # also replace the copy the kernel made at start-up (otherwise the warning keeps appearing)

In [ ]:
# Colab does not include FloPy or the SEAWAT executable (swtv4), so we install them at run time.
# On Colab: FloPy 3.11 is installed and swtv4 goes into /usr/local/bin (gone after a runtime reset).
# On a local PC: your own FloPy is used, and swtv4 is downloaded once into the FloPy bin folder.

import sys, os, shutil
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "flopy<3.12"      # flopy 3.12 needs numpy >= 2.2, which would upgrade Colab's numpy and break numba

import flopy
bindir = Path("/usr/local/bin") if IN_COLAB else Path(os.environ.get("LOCALAPPDATA", Path.home())) / "flopy" / "bin"
swt_exe = shutil.which("swtv4") or shutil.which(str(bindir / "swtv4"))
if swt_exe is None:                   # download swtv4 only if it is not there yet
    bindir.mkdir(parents=True, exist_ok=True)
    flopy.utils.get_modflow(str(bindir), subset="swtv4", quiet=True)
    swt_exe = shutil.which(str(bindir / "swtv4"))
print("FloPy", flopy.__version__, "| swtv4:", swt_exe)  # confirm that both are installed


In [ ]:
from pathlib import Path                 # file paths
from tempfile import TemporaryDirectory  # temporary working folder

In [ ]:
import numpy as np    # array computations

In [ ]:
import flopy          # write SEAWAT input files, run the model, read the results

## The problem
- Confined aquifer, 1,000 m long and 30 m thick. Sea on the **left** (x = 0), land on the right. The sea boundary sits in one extra column to the left of the aquifer (x = −4 to 0 m).
- Inland freshwater inflow 0.15 m³/day, recharge 5 × 10⁻⁵ m/day on the top.
- Stress period 1: the aquifer starts full of seawater and fresh water pushes it back to a steady wedge with the sea at 30 m. Stress period 2: the sea is at 34 m for 80,000 days.
- Paper results: toe at **382 m** before the rise, **432 m at about 8,000 days**, back to **382 m** at 80,000 days. Inland head 31 → about 35 m within about 2,000 days.
- This model: about **405 m → 452 m at about 8,500 days → 405 m**. The wedge moves about 47 m and comes back, as in the paper.

## Step 1. Input variables
Every number that comes from the paper is left blank. Type the value after each `=`. It is given in the **Enter** column of the table just above the cell and after `→` in the comment. The note numbers `[n]` in the code match the tables.

**1-1. Grid** (paper Section 3)

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Note</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Variable</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Enter</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Unit</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;">[1]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>Lx</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">1000</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">m</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Length of the aquifer (sea at x = 0, land at x = Lx)</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">[2]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;"><code>Lz</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">30</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">m</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">Thickness of the confined aquifer (bottom 0 m, top 30 m)</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;">[3]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>ncol</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">251</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">-</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Number of columns: 250 aquifer columns (Δx = 4 m) + 1 sea column on the left (use 126 for a quick test)</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">[4]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;"><code>nlay</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">75</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">-</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">Number of layers → Δz = 0.4 m (use 30 for a quick test)</td></tr>
</tbody></table>

In [ ]:
Lx =          # [1] → 1000.0
Lz =          # [2] → 30.0
ncol =        # [3] → 251  (250 aquifer columns + 1 sea column; quick test: 126)
nlay =        # [4] → 75  (quick test: 30)
nrow, delc = 1, 1.0
delr, delv = Lx / (ncol - 1), Lz / nlay   # column 0 is the sea, outside the aquifer
botm = np.linspace(Lz - delv, 0.0, nlay)
xc = delr * (np.arange(ncol) - 0.5)        # cell-centre distance from the sea (m); sea column at x = -2 m
zc = Lz - delv * (np.arange(nlay) + 0.5)   # cell-centre elevation (m)

**1-2. Sea level**

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Note</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Variable</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Enter</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Unit</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;">[5]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>sl_initial</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">30</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">m</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Sea level before the rise (equal to the top of the aquifer)</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">[6]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;"><code>sl_final</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">34</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">m</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">Sea level after the instantaneous rise (+4 m, a worst case in the paper)</td></tr>
</tbody></table>

In [ ]:
sl_initial =        # [5] → 30.0
sl_final =          # [6] → 34.0

**1-3. Fresh water fluxes** (paper Table 1)

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Note</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Variable</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Enter</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Unit</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;">[7]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>Qinland</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">0.15</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">m³/day</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Regional inflow through the inland boundary (0.005 m³/day per metre of depth × 30 m)</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">[8]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;"><code>rech</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">5e-5</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">m/day</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">Recharge on the top. Q and W stay the same after the rise, which is the key assumption of the paper</td></tr>
</tbody></table>

In [ ]:
Qinland =        # [7] → 0.15
rech =           # [8] → 5e-5

**1-4. Aquifer properties** (paper Table 1)

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Note</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Variable</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Enter</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Unit</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;">[9]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>hk</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">10</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">m/day</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Hydraulic conductivity, isotropic (Henry: 864)</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">[10]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;"><code>ss</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">0.008</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">1/m</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">Specific storage. Large on purpose; it controls how far and how long the wedge moves</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">[11]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;"><code>prsity</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">0.35</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">-</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">Porosity (same as Henry)</td></tr>
</tbody></table>

In [ ]:
hk =            # [9] → 10.0
ss =            # [10] → 0.008
prsity =        # [11] → 0.35

**1-5. Transport properties** (paper Table 1)

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Note</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Variable</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Enter</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Unit</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;">[12]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>al</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">1.0</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">m</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Longitudinal dispersivity α<sub>L</sub> (Henry: 0)</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">[13]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;"><code>trpt</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">0.1</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">-</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">Ratio α<sub>T</sub>/α<sub>L</sub>. The paper gives α<sub>T</sub> = 0.1 m, so 0.1 / 1.0</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">[14]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;"><code>dmcoef</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">0</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">m²/day</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">Molecular diffusion. Not given in the paper, assumed 0 (Henry: 0.57024)</td></tr>
</tbody></table>

In [ ]:
al =            # [12] → 1.0
trpt =          # [13] → 0.1
dmcoef =        # [14] → 0.0

**1-6. Seawater**

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Note</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Variable</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Enter</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Unit</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;">[15]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>csalt</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">35</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">kg/m³</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Seawater concentration</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">[16]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;"><code>densesalt</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">1025</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">kg/m³</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">Seawater density. With 1000 kg/m³ for fresh water it gives the slope 0.7143 and δ = 0.025</td></tr>
</tbody></table>

In [ ]:
csalt =            # [15] → 35.0
densesalt =        # [16] → 1025.0
denseref = 1000.0
denseslp = (densesalt - denseref) / csalt   # 0.7143
delta = (densesalt - denseref) / denseref   # 0.025

**1-7. Simulation time**

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Note</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Variable</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Enter</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Unit</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;">[17]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>t_spinup</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">300000</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">day</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Spin-up with the sea at 30 m, starting from an aquifer full of seawater. Not in the paper; it only has to be long enough for a steady wedge</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">[18]</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;"><code>t_rise</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">80000</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">day</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">Time after the rise (the paper runs 80,000 days to reach the new steady state)</td></tr>
</tbody></table>

In [ ]:
t_spinup =        # [17] → 300000.0
t_rise =          # [18] → 80000.0

## Step 2. Create the SEAWAT model

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Argument</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Value</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;"><code>workspace</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>chang2011_ws</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Normal folder instead of a temporary one, because the run takes several minutes</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;"><code>name</code>, <code>ipakcb</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;"><code>"seawat_slr"</code>, 53</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">Model name and unit number for the cell by cell flows (as Henry)</td></tr>
</tbody></table>

In [ ]:
workspace = Path("chang2011_ws"); workspace.mkdir(exist_ok=True)
name, ipakcb = "seawat_slr", 53
swt = flopy.seawat.Seawat(name, exe_name=swt_exe, model_ws=str(workspace))

## Step 3. DIS package: grid and time

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Argument</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Value</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;"><code>nper</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">2</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Stress period 1 = spin-up (sea 30 m), 2 = after the rise (sea 34 m). Henry: 1</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;"><code>perlen</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;"><code>[t_spinup, t_rise]</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">Length of each stress period [17], [18]</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;"><code>nstp</code>, <code>tsmult</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>[1, 150]</code>, <code>[1.0, 1.03]</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Flow time steps. After the rise they start at about 30 days and grow</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;"><code>steady</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;"><code>[True, False]</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">Steady spin-up, transient rise (storage <code>ss</code> acts only here)</td></tr>
</tbody></table>

In [ ]:
nstp_rise = 150
dis = flopy.modflow.ModflowDis(swt, nlay, nrow, ncol, nper=2, delr=delr, delc=delc, top=Lz, botm=botm,
                               perlen=[t_spinup, t_rise], nstp=[1, nstp_rise], tsmult=[1.0, 1.03],
                               steady=[True, False])

## Step 4. BAS and CHD packages: the sea boundary
The sea is given by CHD instead of `ibound = -1`, so its head can change between stress periods. The heads are equivalent freshwater heads of a seawater column, $h_f = SL + \delta (SL - z)$.

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Argument</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Value</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;"><code>ibound</code>, <code>strt</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">1, 31.0</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">All cells active; starting head is only a first guess</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">CHD cells</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">column 0, all layers</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">Extra sea column left of the aquifer (x = −4 to 0 m), so the aquifer keeps its full 1,000 m (Henry: right column)</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;"><code>shead</code> = <code>ehead</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;"><code>hf(sl_initial)</code>, <code>hf(sl_final)</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">Sea head in stress periods 1 and 2 [5], [6]. Equal start and end values = instantaneous rise</td></tr>
</tbody></table>

In [ ]:
bas = flopy.modflow.ModflowBas(swt, ibound=1, strt=31.0)
hf = lambda sl: sl + delta * (sl - zc)   # freshwater head of the sea column in each layer
chd_data = {per: [[k, 0, 0, hf(sl)[k], hf(sl)[k]] for k in range(nlay)]
            for per, sl in enumerate([sl_initial, sl_final])}
chd = flopy.modflow.ModflowChd(swt, stress_period_data=chd_data)

## Step 5. LPF, PCG and OC packages

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Argument</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Value</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;"><code>laytyp</code>, <code>ss</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">0, <code>ss</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Confined layers with specific storage [10]</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;"><code>rclose</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">1e-8</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">Small, because the fluxes are small</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">OC</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">every step of period 2</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">Heads after every time step, for the inland head time series</td></tr>
</tbody></table>

In [ ]:
lpf = flopy.modflow.ModflowLpf(swt, laytyp=0, hk=hk, vka=hk, ss=ss, ipakcb=ipakcb)
pcg = flopy.modflow.ModflowPcg(swt, mxiter=100, iter1=200, hclose=1e-6, rclose=1e-8)
oc_spd = {(0, 0): ["save head"]} | {(1, k): ["save head"] for k in range(nstp_rise)}
oc = flopy.modflow.ModflowOc(swt, stress_period_data=oc_spd, compact=True)

## Step 6. WEL, RCH and SSM data

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Argument</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Value</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;">WEL</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>Qinland / nlay</code> in the last column</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Inland inflow [7] at x = 996–1,000 m (Henry: first column)</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">RCH</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;"><code>rech</code> on the top layer</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">Recharge [8] on the 250 aquifer columns, none on the sea column</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">SSM</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">WEL 0, CHD <code>csalt</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">Concentration of the inflows [15] (Henry: BAS6 type)</td></tr>
</tbody></table>

In [ ]:
itype = flopy.mt3d.Mt3dSsm.itype_dict()
wel = flopy.modflow.ModflowWel(swt, stress_period_data={0: [[k, 0, ncol - 1, Qinland / nlay] for k in range(nlay)]},
                               ipakcb=ipakcb)
rech_arr = np.full((nrow, ncol), rech); rech_arr[:, 0] = 0.0
rch = flopy.modflow.ModflowRch(swt, nrchop=1, rech=rech_arr, ipakcb=ipakcb)
ssm_sp = ([[k, 0, ncol - 1, 0.0, itype["WEL"]] for k in range(nlay)] +
          [[k, 0, 0, csalt, itype["CHD"]] for k in range(nlay)])

## Step 7. MT3DMS packages

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Argument</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Value</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;"><code>sconc</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">35 everywhere</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">The aquifer starts full of seawater and the spin-up pushes it back (as Henry)</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;"><code>timprs</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">list of times</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">Output times: every 10,000 d in the spin-up, every 250–2,000 d after the rise</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;"><code>dt0</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;"><code>[25, 25]</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Transport step (day). Keep it the same in both periods: with a larger spin-up step the wedge settles in a different place and then drifts after the rise</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;"><code>nadvfd</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">1</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:none;">Upstream finite difference. Central (2) gives oscillations at Δx = 4 m and a result that depends on the time step. The paper's central option is <code>mfnadvfd</code> in Step 8</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">DSP</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;"><code>al</code>, <code>trpt</code>, <code>dmcoef</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:1.5px solid currentColor;">Dispersion [12]–[14]</td></tr>
</tbody></table>

In [ ]:
X, Z = np.meshgrid(xc, zc)
sconc = np.full((nlay, nrow, ncol), csalt)   # aquifer full of seawater at the start
t_out = np.r_[np.arange(250, 5000, 250), np.arange(5000, 20000, 500), np.arange(20000, t_rise + 1, 2000)]
timprs = np.r_[np.linspace(10000, t_spinup, 10), t_spinup + t_out]

btn = flopy.mt3d.Mt3dBtn(swt, nprs=len(timprs), timprs=timprs, prsity=prsity, sconc=sconc, ifmtcn=0,
                         chkmas=False, nprobs=10, nprmas=10, dt0=[25.0, 25.0], mxstrn=100000)
adv = flopy.mt3d.Mt3dAdv(swt, mixelm=0, nadvfd=1)   # upstream finite difference
dsp = flopy.mt3d.Mt3dDsp(swt, al=al, trpt=trpt, trpv=trpt, dmcoef=dmcoef)
gcg = flopy.mt3d.Mt3dGcg(swt, iter1=500, mxiter=1, isolve=2, cclose=1e-6)  # isolve=2 (SSOR): MIC (3) fails to converge on Colab
ssm = flopy.mt3d.Mt3dSsm(swt, stress_period_data={0: ssm_sp, 1: ssm_sp}, crch={0: 0.0}, mxss=1000)

## Check the grid and boundary conditions
Cross-section of the model grid before the run. Blue: constant head (sea). Red: wells (inland inflow). Every cell starts with seawater (35 kg/m³).

In [ ]:
import matplotlib.pyplot as plt

xe = delr * (np.arange(ncol + 1) - 1)          # column edges (m); sea column at x = -4 to 0
ze = np.r_[Lz, botm]                           # layer edges (m)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.fill_between([xe[0], xe[1]], 0, Lz, color="blue")                          # CHD cells (sea)
ax.plot(np.full(nlay, xc[-1]), zc, "o", color="red", ms=3)                     # WEL cells
ax.vlines(xe, 0, Lz, color="0.6", lw=0.3)                                      # grid
ax.hlines(ze, xe[0], xe[-1], color="0.6", lw=0.3)
ax.set(xlim=(xe[0], xe[-1]), ylim=(0, Lz), xlabel="X (m)", ylabel="Z (m)",
       title=f"Row 1  |  {nlay} layers x {ncol} columns  |  CHD {sl_initial:g} -> {sl_final:g} m,  "
             f"WEL {Qinland:g} m3/d,  RCH {rech:g} m/d,  initial conc. {csalt:g}")
plt.show()


## Step 8. VDF package

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Argument</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Value</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;"><code>denseslp</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">0.7143</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">From [15], [16]</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;"><code>mfnadvfd</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">2</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">Central weighting of the density terms (the paper's "central difference" option)</td></tr>
</tbody></table>

In [ ]:
vdf = flopy.seawat.SeawatVdf(swt, iwtable=0, densemin=0, densemax=0, denseref=denseref,
                             denseslp=denseslp, mfnadvfd=2, firstdt=1.0)

## Step 9. Write the input files and run SEAWAT
Expect about 10–15 minutes on Colab (about 7 minutes on a desktop PC). For a first try, use `ncol = 126` and `nlay = 30` in Step 1 (about 2 minutes).

In [ ]:
swt.write_input()
success, buff = swt.run_model(silent=True, report=True)
assert success, "SEAWAT did not terminate normally."

## Step 10. Concentrations and toe position

<table style="width:100%; table-layout:auto; border-collapse:collapse; margin:10px 0 16px 0; font-size:0.95em;">
<thead><tr>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Argument</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Value</th>
<th style="background:rgba(127,127,127,0.30); font-family:Arial, Helvetica, sans-serif; font-weight:700; text-align:center; vertical-align:middle; padding:6px 10px; border:none;">Meaning</th>
</tr></thead><tbody>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:transparent; border-bottom:none;"><code>t_c</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">time − <code>t_spinup</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:transparent; border-bottom:none;">Time since the rise (negative during the spin-up)</td></tr>
<tr><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:center; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;"><code>xt</code></td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">toe position</td><td style="font-family:'Times New Roman', Times, serif; padding:5px 10px; border:none; text-align:left; vertical-align:top; background:rgba(127,127,127,0.13); border-bottom:1.5px solid currentColor;">Where the 50% isochlor (17.5 kg/m³) meets the base, interpolated between cells</td></tr>
</tbody></table>

In [ ]:
import flopy.utils.binaryfile as bf
import matplotlib.pyplot as plt

ucn = bf.UcnFile(workspace / "MT3D001.UCN", model=swt)
conc_all = ucn.get_alldata()                       # (time, layer, row, column)
t_c = np.array(ucn.get_times()) - t_spinup

c50 = 0.5 * csalt
def toe(c):
    cb = c[-1, 0, :]                               # bottom layer
    j = np.argmax(cb < c50)                        # first column fresher than 50 %
    return xc[j - 1] + (cb[j - 1] - c50) / (cb[j - 1] - cb[j]) * delr

xt = np.array([toe(c) for c in conc_all])
i0 = np.argmin(np.abs(t_c))
print(f"Spin-up check: toe {xt[i0 - 1]:.1f} m at t = {t_c[i0 - 1]:.0f} d, {xt[i0]:.1f} m at t = 0 (paper: 382 m)")

## Step 11. Heads
Mean freshwater head in the inland column over time.

In [ ]:
hds = bf.HeadFile(workspace / f"{name}.hds")
heads = hds.get_alldata()
t_h = np.array(hds.get_times()) - t_spinup
hf_inland = heads[:, :, 0, -1].mean(axis=1)

## Step 12. Toe position and inland head over time (Fig. 4 of the paper)
Paper: 382 m → 432 m at about 8,000 days → 382 m; inland head 31 → 35 m in about 2,000 days.

In [ ]:
rise = t_c >= 0
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(t_c[rise], xt[rise], "o-", ms=3, color="tab:blue")
ax1.set(xlabel="Time since the sea-level rise (day)", ylabel="Toe position $X_T$ (m)", xlim=(0, 40000))
ax2 = ax1.twinx()
ax2.plot(t_h[t_h >= 0], hf_inland[t_h >= 0], color="tab:orange")
ax2.set_ylabel("Inland freshwater head $h_f$ (m)", color="tab:orange")

imax = np.argmax(np.where(rise, xt, -np.inf))
print(f"Toe : {xt[i0]:.0f} m -> max {xt[imax]:.0f} m at {t_c[imax]:.0f} d -> {xt[-1]:.0f} m   (paper: 382 -> 432 at ~8,000 d -> 382)")
print(f"Head: {hf_inland[np.argmin(np.abs(t_h))]:.2f} -> {hf_inland[-1]:.2f} m   (paper: 31 -> 35)")

## Step 13. Shape of the wedge over time (Figs. 2 and 3 of the paper)
50% isochlor at the times of Fig. 3. At 80,000 days it should lie on the initial wedge.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for t, col in zip([0, 3000, 8000, 15000, 80000], ["k", "tab:blue", "tab:red", "tab:green", "tab:gray"]):
    i = np.argmin(np.abs(t_c - t))
    ax.contour(xc, zc, conc_all[i, :, 0, :], levels=[c50], colors=col)
    ax.plot([], [], color=col, label=f"{t_c[i]:,.0f} d")
ax.set(xlim=(0, Lx), ylim=(0, Lz), xlabel="x (m)", ylabel="z (m)", title="50% isochlor")
ax.legend()

## Clean up
Deletes the working folder. Skip it to keep the results.

In [ ]:
import shutil
shutil.rmtree(workspace, ignore_errors=True)

**Reference.** Chang, S. W., Clement, T. P., Simpson, M. J., & Lee, K.-K. (2011). Does sea-level rise have an impact on saltwater intrusion? *Advances in Water Resources*, 34(10), 1283–1291. https://doi.org/10.1016/j.advwatres.2011.06.006